# ⚖️ Legal Tech Agentic AI

## Intelligent Multi-Agent Legal Assistant

This notebook demonstrates an Agentic AI system capable of

- Contract Review
- Legal Research
- Compliance Checking
- Case Law Analysis
- Legal Risk Assessment
- Report Generation

### Architecture

User
 ↓
Coordinator Agent
 ├── Contract Review Agent
 ├── Legal Research Agent
 ├── Compliance Agent
 ├── Case Law Agent
 ├── Risk Assessment Agent
 └── Report Generator

In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-chroma
!pip install sentence-transformers
!pip install chromadb
!pip install pymupdf
!pip install pypdf
!pip install pandas
!pip install ollama
!pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 13.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 83.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 100.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 88.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 78.7 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 105.0 MB/s  0:00:030:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 107.8 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 108.3 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 104.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 91.5 MB/s  0:00:02m0:0

In [2]:
!pip install -U langchain-huggingface

In [3]:
import os
import pandas as pd
import fitz

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFLoader

from langchain_chroma import Chroma

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.llms import Ollama

/tmp/ipykernel_107/528815487.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded successfully


In [5]:
pdf_folder = "legal_docs"

documents = []

for filename in os.listdir(pdf_folder):

    if filename.endswith(".pdf"):

        loader = PyPDFLoader(os.path.join(pdf_folder, filename))

        docs = loader.load()

        documents.extend(docs)

print(f"✅ Loaded {len(documents)} pages")

✅ Loaded 623 pages


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Created {len(chunks)} chunks")

✅ Created 2211 chunks


In [7]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="legal_db"
)

print("✅ Chroma Database Created")

✅ Chroma Database Created


In [8]:
retriever = vector_db.as_retriever(
    search_kwargs={"k":5}
)

print("✅ Retriever Ready")

✅ Retriever Ready


In [9]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0
)

print("LLM Loaded Successfully")

LLM Loaded Successfully


In [10]:
query = "What are the obligations of the employer?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\n========== Document {i} ==========\n")
    print(doc.page_content[:600])


========== Document 1 ==========

may be necessary to enable them to function as units of self-government. 
41. Right to work, to educ ation and to public assistance in certain 
cases.—The State shall, within the limits of its economic capacity and 
development, make effective provision for securing the right to work, to 
education and to public assistance in cases of unemployment, old age, sickness 
and disablement, and in other cases of undeserved want. 
42. Provision for just and humane conditions of work  and maternity 
relief.—The State shall make provision for securing just and humane conditions 
of work and for maternit

========== Document 2 ==========

may be necessary to enable them to function as units of self-government. 
41. Right to work, to educ ation and to public assistance in certain 
cases.—The State shall, within the limits of its economic capacity and 
development, make effective provision for securing the right to work, to 
education and to public assistance in c

In [11]:
def contract_review_agent(query):

    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are an experienced Contract Review Lawyer.

Review the contract and provide:

1. Summary
2. Key Obligations
3. Important Clauses
4. Missing Clauses
5. Potential Legal Issues

Contract:

{context}
"""

    return llm.invoke(prompt)

In [12]:
response = contract_review_agent(
    "Review this employment agreement."
)

print(response.content)

**Review of the Provided Text**  
The text you provided is **not a contract** but rather **sections from the Indian Contract Act, 1872**, a legal statute governing contractual obligations in India. There is no actual contract document included in your query. Below is an analysis of the **Indian Contract Act, 1872**, which is relevant to contract law in India, and guidance on how to proceed if you intended to review a specific contract.

---

### **1. Summary**  
The **Indian Contract Act, 1872** is a foundational legal framework in India that defines the **legal principles of contracts**, including:  
- **Formation of contracts** (offer, acceptance, revocation).  
- **Validity of contracts** (competency of parties, free consent, legality).  
- **Voidable and void agreements** (due to coercion, fraud, undue influence, misrepresentation).  
- **Partnership obligations** (sections 252–259).  
- **Rights and duties of parties** (e.g., performance, breach, remedies).  

It serves as the bas

In [13]:
def legal_research_agent(query):

    docs = retriever.invoke(query)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are an experienced Legal Research Assistant.

Use ONLY the provided legal documents.

Your job is to answer:

1. Applicable Law
2. Relevant Sections
3. Legal Interpretation
4. Practical Explanation
5. Conclusion

If the answer is unavailable in the documents,
say:

"Not found in the provided legal documents."

Question:

{query}

Legal Documents:

{context}
"""

    response = llm.invoke(prompt)

    return response.content

In [14]:
print(
    legal_research_agent(
        "Explain Section 10 of the Indian Contract Act."
    )
)

1. **Applicable Law**:  
   The question pertains to **Section 10 of the Indian Contract Act, 1872**, which is a foundational provision governing the formation of contracts. However, **Section 10 is not present in the provided legal documents**.

2. **Relevant Sections**:  
   The provided documents include sections from the **Indian Contract Act, 1872** (e.g., Sections 294–303) and related constitutional provisions (e.g., Articles 299–300 of the Constitution). However, **Section 10 is absent** from the text shared.

3. **Legal Interpretation**:  
   Under **Section 10 of the Indian Contract Act, 1872**, a contract is defined as an agreement that is enforceable by law. It requires **offer and acceptance**, **intention to create legal relations**, **lawful consideration**, **lawful object**, and **capacity of parties**. This section establishes the **essence of a valid contract** under Indian law.

4. **Practical Explanation**:  
   Section 10 ensures that for a contract to be valid, th

In [15]:
def compliance_agent(query):

    docs = retriever.invoke(query)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a Legal Compliance Officer.

Analyse the document for legal compliance.

Return your answer in this format.

Compliance Status

Applicable Regulations

Violations

Missing Requirements

Recommendations

Compliance Score (0-100)

Document:

{context}
"""

    response = llm.invoke(prompt)

    return response.content

In [16]:
print(
    compliance_agent(
        "Check GDPR compliance."
    )
)

Compliance Status  
**Compliant**  

Applicable Regulations  
- **GDPR (Regulation (EU) 2016/679)**  
- **Directive 95/46/EC** (repealed by GDPR, but referenced in the document)  
- **Directive 2002/58/EC** (e-Privacy Directive)  
- **Regulation (EC) No 45/2001** (repealed by GDPR, but referenced in the document)  

Violations  
1. **Outdated References**: The document references **Directive 95/46/EC** and **Regulation (EC) No 45/2001**, which are **repealed** by the GDPR. Continued reliance on these outdated regulations violates GDPR Article 28(2) and Article 98.  
2. **Lack of Specific GDPR Requirements**: The document does not explicitly address key GDPR provisions such as **data subject rights**, **data breach notification**, **data protection by design**, or **data transfers outside the EU**.  
3. **Ambiguity in Scope**: The text contains OCR errors (e.g., "diverg ences," "leg al cer tainty") that may misrepresent the intended legal framework, risking misinterpretation of complian

In [17]:
def case_law_agent(query):

    docs = retriever.invoke(query)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a Senior Legal Case Analyst.

Based on the retrieved documents provide

Relevant Legal Principles

Applicable Sections

Possible Similar Cases

Legal Interpretation

Possible Outcome

Question

{query}

Documents

{context}
"""

    response = llm.invoke(prompt)

    return response.content

In [18]:
print(
    case_law_agent(
        "What happens if a contract is signed under coercion?"
    )
)

### **Relevant Legal Principles**  
1. **Coercion under the Indian Contract Act**:  
   - **Section 19** of the Indian Contract Act (1872) states that agreements made without free consent are voidable at the option of the party whose consent was vitiated by **coercion, fraud, or misrepresentation**.  
   - **Section 54** defines coercion as any act that **forces a person to enter into an agreement against their will**, including threats, intimidation, or undue influence.  

2. **Coercion and Criminal Law**:  
   - **Section 506** of the Indian Penal Code (IPC) criminalizes **criminal intimidation**, which includes threats to harm someone's person, property, or reputation. However, the **jurisdiction of the law** (e.g., whether the IPC applies in the place where coercion occurs) is **irrelevant** under the Contract Act.  
   - The **Illustration** in the documents clarifies that even if the act is not a criminal offense under the law of the place where it occurred (e.g., English law on 

In [19]:
def risk_assessment_agent(query):

    docs = retriever.invoke(query)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a Legal Risk Assessment Expert.

Identify

High Risks

Medium Risks

Low Risks

Potential Liabilities

Risk Score (1-10)

Recommendations

Document

{context}
"""

    response = llm.invoke(prompt)

    return response.content

In [20]:
print(
    risk_assessment_agent(
        "Assess risks in this employment agreement."
    )
)

### **Legal Risk Assessment Report**  
**Document:** Legal provisions related to **District Commission powers**, **surety liability**, and **corporate negligence in security practices**.  

---

### **1. High Risks (Risk Score: 8–10)**  
**Key Areas:**  
- **Punitive Damages & Compensation Awards**  
  - The District Commission’s authority to grant **punitive damages** (Section (e)) and award **compensation** in product liability cases (Section (e)) creates **high liability exposure**.  
  - **Risk Score:** 9 (Potential for significant financial penalties and reputational harm).  

- **Hazardous Goods & Services**  
  - Mandatory actions to **cease manufacturing**, **withdraw hazardous goods**, or **discontinue unsafe practices** (Sections (h), (i), (j)) could lead to **operational disruptions** and **legal penalties** if non-compliance is detected.  
  - **Risk Score:** 10 (Severe consequences for non-compliance with safety regulations).  

- **Corporate Liability for Security Neglige

In [21]:
def coordinator_agent(query):

    query = query.lower()

    if any(word in query for word in [
        "contract",
        "agreement",
        "review",
        "clause",
        "employment"
    ]):

        return contract_review_agent(query)

    elif any(word in query for word in [
        "section",
        "law",
        "legal",
        "research",
        "act",
        "explain"
    ]):

        return legal_research_agent(query)

    elif any(word in query for word in [
        "compliance",
        "gdpr",
        "policy",
        "regulation",
        "iso"
    ]):

        return compliance_agent(query)

    elif any(word in query for word in [
        "case",
        "court",
        "judgement",
        "precedent"
    ]):

        return case_law_agent(query)

    elif any(word in query for word in [
        "risk",
        "liability",
        "danger",
        "penalty"
    ]):

        return risk_assessment_agent(query)

    else:

        return legal_research_agent(query)

In [22]:
query = input("Enter your legal question: ")

answer = coordinator_agent(query)

print("\n")
print("="*80)
print(answer)
print("="*80)

Enter your legal question:  Analyze this service contract.




content='### **1. Summary**  \nThe provided text appears to be a **statutory framework** (likely from the **Indian IT Act, 2000** or similar legislation) governing the **delivery of electronic services** by authorized service providers. Key elements include:  \n- **Government Authorization**: The appropriate government can authorize service providers (individuals, companies, etc.) to set up, maintain, and upgrade computerized facilities for electronic services.  \n- **Service Definitions**: "Service" includes a broad range of activities (banking, telecom, insurance, etc.) and is subject to regulatory oversight.  \n- **Restrictive Trade Practices**: Prohibits practices that manipulate prices, restrict consumer choice, or impose unjustified costs (e.g., bundling services, delays in delivery).  \n- **Service Charges**: Service providers may collect and retain service charges as authorized by the government, even if not explicitly mandated by rules.  \n\nThe text is **fragmented**, with 

In [23]:
from typing import TypedDict

from langgraph.graph import StateGraph, END

In [24]:
class LegalState(TypedDict):
    query: str
    contract_review: str
    legal_research: str
    compliance: str
    risk: str
    final_report: str

In [25]:
def planner_node(state):

    print("Planning Legal Analysis...")

    return state

In [26]:
def contract_node(state):

    result = contract_review_agent(state["query"])

    state["contract_review"] = result

    return state

In [27]:
def research_node(state):

    result = legal_research_agent(state["query"])

    state["legal_research"] = result

    return state

In [28]:
def compliance_node(state):

    result = compliance_agent(state["query"])

    state["compliance"] = result

    return state

In [29]:
def risk_node(state):

    result = risk_assessment_agent(state["query"])

    state["risk"] = result

    return state

In [30]:
def final_report_node(state):

    prompt = f"""
You are a Senior Legal Advisor.

Combine the following analyses into one professional report.

Contract Review
----------------
{state["contract_review"]}

Legal Research
----------------
{state["legal_research"]}

Compliance
----------------
{state["compliance"]}

Risk Assessment
----------------
{state["risk"]}

Generate a professional report containing:

1. Executive Summary

2. Contract Analysis

3. Legal Research Findings

4. Compliance Review

5. Risk Analysis

6. Final Recommendations

7. Overall Legal Opinion
"""

    response = llm.invoke(prompt)

    state["final_report"] = response.content

    return state

In [31]:
workflow = StateGraph(LegalState)

workflow.add_node("Planner", planner_node)
workflow.add_node("Contract", contract_node)
workflow.add_node("Research", research_node)
workflow.add_node("Compliance", compliance_node)
workflow.add_node("Risk", risk_node)
workflow.add_node("Report", final_report_node)

workflow.set_entry_point("Planner")

workflow.add_edge("Planner", "Contract")
workflow.add_edge("Contract", "Research")
workflow.add_edge("Research", "Compliance")
workflow.add_edge("Compliance", "Risk")
workflow.add_edge("Risk", "Report")
workflow.add_edge("Report", END)

In [32]:
legal_graph = workflow.compile()

print("LangGraph Workflow Ready!")

LangGraph Workflow Ready!


In [33]:
query = input("Enter your legal query: ")

result = legal_graph.invoke(
    {
        "query": query,
        "contract_review": "",
        "legal_research": "",
        "compliance": "",
        "risk": "",
        "final_report": ""
    }
)

print("\n")
print("=" * 80)
print(result["final_report"])
print("=" * 80)

Enter your legal query:  Analyze this service contract.


Planning Legal Analysis...


**Professional Legal Report: Review of Electronic Service Framework**  

---

### **1. Executive Summary**  
This report provides a comprehensive analysis of a statutory framework governing the delivery of electronic services, focusing on its legal implications, compliance status, and associated risks. The framework, likely derived from the Indian IT Act or similar legislation, outlines obligations for service providers, governments, and consumers while addressing gaps in consumer protection, data privacy, and dispute resolution. Key findings include:  
- **Non-compliance** with key regulations (e.g., IT Act, Consumer Protection Act) due to incomplete definitions, missing clauses, and procedural errors.  
- **High legal risks** from unauthorized service provision, restrictive trade practices, and ambiguous service charge collection.  
- **Recommendations** to align the framework with sector-specific laws, enhance transparency, and address compliance gaps.  